In [9]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2
from rate_limiter import rate_limiter

from tqdm.asyncio import tqdm as tqdm_asyncio

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_39255/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [4]:
df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견'],
      dtype='object')

In [5]:
df.columns = df.columns.str.strip()
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt',
       'Lateral excursion Protrusive excursion', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [6]:
df = df[['환자번호', '날짜', 'CC', '약', '장치', '습관', '찜질', '마사지, 스트레칭', 'PI']]

In [8]:
df = df.sample(10)

In [49]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    # MODEL_NAME = "gpt-4o"
    MODEL_NAME = "o1-mini"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"


#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)


#############################################
# 체크포인트 관리 클래스
#############################################

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None


#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

class MedicalTextClassifier:
    """의학 텍스트 분류기 클래스"""
    
    def __init__(self, api_key: str, config=None):
        """초기화"""
        self.config = config if config is not None else Config
        self.client = openai.OpenAI(api_key=api_key)  # 클라이언트 객체 생성 방식 변경
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager(self.config.CHECKPOINT_DIR)
        
        # 분류기 메소드 맵핑
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리"""
        original_shape = df.shape
        processed_cols = 0
        
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
                
                # 처리 후 해당 컬럼에서 파생된 새 컬럼 확인
                derived_cols = [col for col in df.columns if col.startswith(f"{column}_")]
                logger.info(f"Column {column} generated {len(derived_cols)} derived columns: {derived_cols}")
                
                # 파생 컬럼의 값이 있는 행 수 확인
                for derived_col in derived_cols:
                    non_empty_count = df[derived_col].notna().sum()
                    logger.info(f"Column {derived_col} has {non_empty_count} non-empty values")
                
                processed_cols += 1
        
        logger.info(f"Original DataFrame shape: {original_shape}, Processed columns: {processed_cols}")
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트를 사용한 컬럼 처리"""
        try:
            # 기존 체크포인트 확인
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                # 체크포인트로부터 원본 데이터프레임 업데이트
                for idx in checkpoint_df.index:
                    if idx in df.index:
                        for col in checkpoint_df.columns:
                            # 원본 DataFrame에 직접 값 설정
                            df.loc[idx, col] = checkpoint_df.loc[idx, col]
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # 유효한 텍스트 필터링 (null이 아니고 길이가 1보다 큰 경우만)
            mask = df[column].notna() & (df[column].str.len() > 1) & df[column].str.strip().astype(bool)
            
            if not mask.any():
                logger.info(f"No valid text entries found in column {column}")
                return df

            # 필터링된 텍스트와 인덱스 쌍 생성
            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            
            # 로깅 추가: 처리 대상 텍스트 출력
            for idx, text in texts_with_idx:
                logger.info(f"Processing text at index {idx}, first 100 chars: {text[:100]}")
            
            # 데이터 처리 - 여기서 results 변수가 정의됨
            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            # 결과 처리
            if results:
                result_df = pd.DataFrame(results).set_index('index')
                logger.info(f"Result DataFrame shape: {result_df.shape}, indices: {result_df.index.tolist()}")
                
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    
                    # 각 인덱스에 명시적으로 할당
                    for idx in result_df.index:
                        if idx in df.index:
                            logger.debug(f"Setting value at index {idx}, column {new_col}: {result_df.loc[idx, col]}")
                            df.loc[idx, new_col] = result_df.loc[idx, col]
                        else:
                            logger.warning(f"Index {idx} from result not found in DataFrame")

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                classifier, column: str) -> List[Dict]:
        """더 높은 신뢰성을 위해 한 번에 하나의 레코드 처리"""
        results = []

        for i, (text, idx) in enumerate(zip(texts, original_indices)):
            try:
                # 추가 확인: 실제 처리 직전에 텍스트가 유효한지 재확인
                if text is None or len(str(text).strip()) <= 1:
                    logger.info(f"Skipping invalid text at index {idx} (None or length ≤ 1)")
                    # 인덱스 매핑을 유지하기 위해 빈 결과 추가
                    results.append({"index": idx})
                    continue
                    
                logger.info(f"단일 텍스트 처리 중 {i+1}/{len(texts)} (인덱스 {idx})")
                
                # 하나의 텍스트만 처리
                batch_result = await self._process_with_retry(classifier, [text], [idx])
                
                if batch_result:
                    results.extend(batch_result)
                    
                    # 이 결과만으로 DataFrame 생성
                    partial_df = pd.DataFrame(batch_result).set_index('index')
                    self.checkpoint.save_checkpoint(partial_df, column)
                    
            except Exception as e:
                logger.error(f"인덱스 {idx}의 텍스트 처리 실패: {str(e)}")
                # 인덱스 매핑을 유지하기 위해 빈 결과 추가
                results.append({"index": idx})

        return results

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
# _process_with_retry 메서드에 추가
    async def _process_with_retry(self, classifier, batch_texts: List[str], batch_indices: List[int]) -> List[Dict]:
        """각 텍스트를 개별적으로 처리하여 정확한 매핑 보장"""
        results = []

        for i, text in enumerate(batch_texts):
            async with self.semaphore:
                # 한 번에 하나의 텍스트 처리
                single_result = await classifier([text], self.semaphore)
                if single_result and len(single_result) > 0:
                    # 결과에 인덱스 추가
                    results.append({"index": batch_indices[i], **single_result[0]})
                else:
                    # API가 빈 또는 유효하지 않은 결과를 반환한 경우 처리
                    results.append({"index": batch_indices[i]})
                
                # 각 개별 결과 로깅
                logger.info(f"텍스트 {i} (인덱스 {batch_indices[i]}) 처리 완료: {single_result[0] if single_result and len(single_result) > 0 else '결과 없음'}")
        
        return results

    def _cleanup_checkpoint(self, column: str) -> None:
        """성공적인 처리 후 체크포인트 정리"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 메소드 (OpenAI API v1.0.0+)"""
        try:
            async with semaphore:
                client = openai.OpenAI(api_key=self.config.API_KEY)
                response = await asyncio.to_thread(
                    client.chat.completions.create,
                    model=self.config.MODEL_NAME,
                    messages=[
                        {"role": "system", "content": "JSON 형식으로 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=self.config.MAX_TOKENS,
                    temperature=self.config.TEMPERATURE
                )
                content = response.choices[0].message.content
                logger.debug(f"API Response: {content[:200]}...")
                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """향상된 JSON 응답 검증 및 파싱"""
        try:
            # 원시 내용 로깅
            logger.info(f"원시 API 응답: {content[:500]}...")
            
            # 먼저 직접 JSON 파싱 시도
            try:
                parsed = json.loads(content)
                if isinstance(parsed, list):
                    return parsed
            except json.JSONDecodeError:
                pass  # 직접 파싱이 실패하면 정규식 추출로 계속 진행
            
            # 정규식으로 JSON 추출
            json_pattern = r'```json\s*([\s\S]*?)\s*```|(\[[\s\S]*\])'
            matches = re.findall(json_pattern, content)
            
            for match in matches:
                # 각 일치 항목 시도
                for m in match:
                    if not m.strip():
                        continue
                        
                    try:
                        parsed = json.loads(m.strip())
                        if isinstance(parsed, list):
                            logger.info(f"JSON 파싱 성공: {parsed}")
                            return parsed
                    except:
                        continue
            
            # 마지막 수단: JSON 객체나 배열처럼 보이는 것 찾기
            fallback_pattern = r'(\{[\s\S]*?\}|\[[\s\S]*?\])'
            fallback_matches = re.findall(fallback_pattern, content)
            
            for m in fallback_matches:
                try:
                    parsed = json.loads(m.strip())
                    if isinstance(parsed, dict):
                        # 단일 객체를 목록으로 변환
                        return [parsed]
                    elif isinstance(parsed, list):
                        return parsed
                except:
                    continue
                    
            logger.error(f"응답에서 JSON을 파싱하지 못함")
            return []
            
        except Exception as e:
            logger.error(f"JSON 파싱 오류: {str(e)}")
            return []
        
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Chief Complaints 분류"""
        # CC를 3개의 작은 프롬프트로 분할
        cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
        history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
        severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
        
        # 결과 병합
        combined_results = []
        for i in range(len(texts)):
            combined_dict = {}
            if i < len(cc_results):
                combined_dict.update(cc_results[i])
            if i < len(history_results):
                combined_dict.update(history_results[i])
            if i < len(severity_results):
                combined_dict.update(severity_results[i])
            
            combined_results.append(combined_dict)
        
        return combined_results
    
    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        results = await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        logger.debug(f"약물 분류 결과: {results}")
        return results

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        results = await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        logger.debug(f"장치 분류 결과: {results}")
        return results

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        results = await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        logger.debug(f"습관 분류 결과: {results}")
        return results

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        results = await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        logger.debug(f"찜질 분류 결과: {results}")
        return results
    
    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지 및 스트레칭 분류"""
        results = await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        logger.debug(f"마사지 분류 결과: {results}")
        return results 
        
    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """PI 분류 개선"""
        # 유효하지 않은 텍스트 필터링 (None, 비어있거나 길이가 1 이하)
        valid_texts = []
        valid_indices = []
        result_mapping = {}
        
        for i, text in enumerate(texts):
            if text is None or len(str(text).strip()) <= 1:
                # 유효하지 않은 텍스트는 빈 결과로 매핑
                result_mapping[i] = {}
            else:
                # 텍스트 길이 제한 (10,000자)
                valid_text = text[:10000] if len(text) > 10000 else text
                valid_texts.append(valid_text)
                valid_indices.append(i)
        
        # 유효한 텍스트가 없으면 빈 결과 반환
        if not valid_texts:
            logger.info("No valid texts for PI classification")
            return [{} for _ in range(len(texts))]
        
        # 로깅 추가: 유효한 텍스트 확인
        for i, text in enumerate(valid_texts):
            logger.info(f"Valid text {i}, length {len(text)}, first 100 chars: {text[:100]}")
        
        smaller_batch_size = 10  # PI에 대해 더 작은 배치 크기 사용
        all_valid_results = []
        
        for i in range(0, len(valid_texts), smaller_batch_size):
            batch = valid_texts[i:i+smaller_batch_size]
            batch_results = []  # 각 배치의 결과를 저장할 리스트
            
            # 로깅 추가: 배치 크기 및 첫 번째 텍스트 확인
            logger.info(f"Processing batch {i//smaller_batch_size + 1}, size: {len(batch)}")
            if batch:
                logger.info(f"First item in batch, length {len(batch[0])}, first 100 chars: {batch[0][:100]}")
        
            try:
                logger.info(f"Processing PI basic info batch {i//smaller_batch_size + 1}")
                pi_aggravating_factors_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_aggravating_factors(batch), semaphore)
                
                logger.info(f"Processing PI examination batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_drug_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_drug_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_closing_dentalgear_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_closing_dentalgear_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_check_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_check(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_diagnosis_jojint_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_diagnosis_jojint(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_occlusal_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_occlusal_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_medication_prescription_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_medication_prescription(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_other_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_other_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_onset_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_onset(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_pattern_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_pattern(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_status_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_status(batch), semaphore)
                
                for j in range(len(batch)):
                    combined_dict = {}
                    if j < len(pi_aggravating_factors_results):
                        combined_dict.update(pi_aggravating_factors_results[j])
                    if j < len(pi_TMJ_PI_desc_results):
                        combined_dict.update(pi_TMJ_PI_desc_results[j])
                    if j < len(pi_TMJ_PI_treatment_results):
                        combined_dict.update(pi_TMJ_PI_treatment_results[j])
                    if j < len(pi_drug_treatment_results):
                        combined_dict.update(pi_drug_treatment_results[j])
                    if j < len(pi_closing_dentalgear_desc_results):
                        combined_dict.update(pi_closing_dentalgear_desc_results[j])
                    if j < len(pi_PI_check_results):
                        combined_dict.update(pi_PI_check_results[j])
                    if j < len(pi_PI_diagnosis_jojint_results):
                        combined_dict.update(pi_PI_diagnosis_jojint_results[j])
                    if j < len(pi_occlusal_treatment_results):
                        combined_dict.update(pi_occlusal_treatment_results[j])
                    if j < len(pi_medication_prescription_results):
                        combined_dict.update(pi_medication_prescription_results[j])
                    if j < len(pi_other_treatment_results):
                        combined_dict.update(pi_other_treatment_results[j])
                    if j < len(pi_onset_results):
                        combined_dict.update(pi_onset_results[j])
                    if j < len(pi_pattern_results):
                        combined_dict.update(pi_pattern_results[j])
                    if j < len(pi_status_results):
                        combined_dict.update(pi_status_results[j])
                    
                    batch_results.append(combined_dict)
                
                all_valid_results.extend(batch_results)
            
            except Exception as e:
                logger.error(f"Error processing PI batch {i//smaller_batch_size + 1}: {str(e)}")
                all_valid_results.extend([{} for _ in range(len(batch))])
        
        # 최종 결과 배열 생성 (원래 순서대로)
        # 먼저 빈 결과 딕셔너리로 채움
        final_results = [{} for _ in range(len(texts))]
        
        # 유효한 텍스트에 대한 결과를 올바른 위치에 넣음
        for valid_idx, result in zip(valid_indices, all_valid_results):
            final_results[valid_idx] = result
        
        return final_results

        
#############################################
# 의학 데이터 처리 함수
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        processed_df = await classifier.process_all_columns(df)

        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise


#############################################
# 메인 함수
#############################################

#############################################
# 메인 함수
#############################################

def main():
    """메인 함수"""
    global logger
    logger = setup_logging()
    total_start_time = datetime.now()

    try:
        # 실제 환자 데이터를 로드하는 경우 아래 주석을 해제하세요.
        # df = pd.read_excel('환자데이터.xlsx')
        
        # 원본 데이터 불러오기
        original_df = df
        chunk_size = 2000
        total_rows = len(original_df)
        
        # df_checkpoint 폴더 생성
        checkpoint_dir = 'df_checkpoint'
        if not os.path.exists(checkpoint_dir):
            os.makedirs(checkpoint_dir)
            logger.info(f"체크포인트 디렉토리 '{checkpoint_dir}'를 생성했습니다.")
        
        # 체크포인트 파일 확인
        checkpoint_file = os.path.join(checkpoint_dir, 'checkpoint.json')
        last_processed_chunk = -1
        all_processed_df = pd.DataFrame()
        
        if os.path.exists(checkpoint_file):
            try:
                with open(checkpoint_file, 'r', encoding='utf-8') as f:
                    checkpoint_data = json.load(f)
                last_processed_chunk = checkpoint_data.get('last_processed_chunk', -1)
                
                # 이미 처리된 데이터 불러오기
                processed_file = checkpoint_data.get('processed_file', '')
                if os.path.exists(processed_file):
                    logger.info(f"체크포인트 파일 {checkpoint_file}을 찾았습니다. 이전 진행 상황을 불러옵니다.")
                    logger.info(f"이미 {last_processed_chunk + 1}개 청크가 처리되었습니다.")
                    
                    # 파일 형식에 따라 적절한 로드 방법 사용
                    if processed_file.endswith('.parquet'):
                        try:
                            all_processed_df = pd.read_parquet(processed_file)
                        except Exception as e:
                            logger.warning(f"Parquet 파일 로드 실패: {str(e)}. CSV 로드를 시도합니다.")
                            csv_file = processed_file.replace('.parquet', '.csv')
                            if os.path.exists(csv_file):
                                all_processed_df = pd.read_csv(csv_file, index_col=0, encoding='utf-8-sig')
                            else:
                                raise Exception(f"대체 CSV 파일을 찾을 수 없습니다: {csv_file}")
                    elif processed_file.endswith('.csv'):
                        all_processed_df = pd.read_csv(processed_file, index_col=0, encoding='utf-8-sig')
                    else:
                        raise Exception(f"지원되지 않는 파일 형식: {processed_file}")
                    
                    # 데이터 로드 확인
                    logger.info(f"체크포인트에서 {len(all_processed_df)}개 행, {len(all_processed_df.columns)}개 열 로드 완료")
            except Exception as e:
                logger.warning(f"체크포인트 파일 로드 실패: {str(e)}. 처음부터 다시 시작합니다.")
                last_processed_chunk = -1
                all_processed_df = pd.DataFrame()
        
        # 청크 생성
        chunks = [original_df[i:i+chunk_size] for i in range(0, total_rows, chunk_size)]
        remaining_chunks = chunks[last_processed_chunk + 1:]
        
        logger.info(f"원본 데이터 총 {total_rows}개를 {len(chunks)}개 청크로 분할하여 처리합니다. (청크 크기: {chunk_size})")
        if last_processed_chunk >= 0:
            logger.info(f"체크포인트에서 재시작: {len(remaining_chunks)}개 청크 남음")
        
        api_key = Config.API_KEY
        
        # MedicalTextClassifier 인스턴스 생성
        classifier = MedicalTextClassifier(api_key)
        
        # 실패한 레코드를 저장할 데이터프레임
        failed_records = pd.DataFrame(columns=['index', 'column', 'text', 'error'])
        
        loop = asyncio.get_event_loop()
        
        # tqdm 진행바 추가
        from tqdm import tqdm
        
        for chunk_num, chunk_df in tqdm(enumerate(remaining_chunks, start=last_processed_chunk + 1), 
                                        total=len(remaining_chunks),
                                        desc="청크 처리 진행률"):
            logger.info(f"청크 {chunk_num+1}/{len(chunks)} 처리 시작 (레코드 {chunk_num*chunk_size+1}-{min((chunk_num+1)*chunk_size, total_rows)})")
            
            try:
                # 청크 처리
                processed_chunk = loop.run_until_complete(process_medical_data(chunk_df, api_key))
                
                # 전체 결과에 추가
                if all_processed_df.empty:
                    all_processed_df = processed_chunk
                else:
                    all_processed_df = pd.concat([all_processed_df, processed_chunk], ignore_index=False)
                
                # 실패한 레코드 확인 (처리된 결과가 없는 행)
                for column in processed_chunk.columns:
                    if column in classifier.classifiers.keys():
                        related_columns = [col for col in processed_chunk.columns if col.startswith(f"{column}_")]
                        
                        if related_columns:  # 파생 컬럼이 존재하는 경우
                            # NaN 값이 있는 행 = 처리 실패한 행
                            mask = processed_chunk[column].notna() & processed_chunk[column].str.strip().astype(bool)
                            has_data_mask = mask.copy()
                            
                            for related_col in related_columns:
                                has_data_mask = has_data_mask & processed_chunk[related_col].isna()
                            
                            failed_indices = processed_chunk[has_data_mask].index.tolist()
                            
                            for idx in failed_indices:
                                failed_records = pd.concat([failed_records, pd.DataFrame([{
                                    'index': idx,
                                    'column': column,
                                    'text': processed_chunk.loc[idx, column],
                                    'error': '처리 결과 없음'
                                }])], ignore_index=True)
                                logger.warning(f"레코드 처리 실패: 인덱스 {idx}, 컬럼 {column}, 텍스트: {processed_chunk.loc[idx, column][:100]}...")
                
                # 각 청크 처리 후 체크포인트 저장
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                
                # 타입 충돌 해결을 위한 데이터프레임 사본 생성
                df_to_save = all_processed_df.copy()
                
                # 데이터 타입 문제 해결 - 모든 컬럼에 대해 안전한 변환 적용
                for col in df_to_save.columns:
                    if col.endswith('_pi_next_schedule'):
                        # 정수형 컬럼은 모두 문자열로 변환
                        df_to_save[col] = df_to_save[col].astype(str)
                    elif df_to_save[col].dtype == 'object':
                        # 객체 타입의 컬럼에서 None이나 nan 값 처리
                        df_to_save[col] = df_to_save[col].fillna('').astype(str)
                
                # 파일 저장
                temp_file = os.path.join(checkpoint_dir, f'temp_processed_data_{timestamp}.parquet')
                try:
                    df_to_save.to_parquet(temp_file)
                except Exception as e:
                    logger.warning(f"Parquet 저장 실패: {str(e)}. CSV 형식으로 저장합니다.")
                    csv_file = os.path.join(checkpoint_dir, f'temp_processed_data_{timestamp}.csv')
                    df_to_save.to_csv(csv_file, index=True, encoding='utf-8-sig')
                    temp_file = csv_file  # 체크포인트 파일 경로 업데이트
                
                checkpoint_data = {
                    'last_processed_chunk': chunk_num,
                    'processed_file': temp_file,
                    'timestamp': timestamp
                }
                
                with open(checkpoint_file, 'w') as f:
                    json.dump(checkpoint_data, f)
                
                logger.info(f"체크포인트 저장 완료: 청크 {chunk_num+1}/{len(chunks)}")
            
            except Exception as e:
                logger.error(f"청크 {chunk_num+1} 처리 중 오류 발생: {str(e)}")
                # 오류가 발생한 청크의 모든 행을 실패로 기록
                for idx, row in chunk_df.iterrows():
                    for column in row.index:
                        if column in classifier.classifiers.keys() and pd.notna(row[column]) and str(row[column]).strip():
                            failed_records = pd.concat([failed_records, pd.DataFrame([{
                                'index': idx,
                                'column': column,
                                'text': row[column],
                                'error': str(e)
                            }])], ignore_index=True)
                
                # 오류 발생 시 지금까지 처리된 데이터 저장
                if not all_processed_df.empty:
                    error_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    
                    # 데이터 타입 문제 해결을 위한 복사본 생성
                    df_error_save = all_processed_df.copy()
                    
                    # 모든 컬럼을 문자열로 변환하여 타입 문제 해결
                    for col in df_error_save.columns:
                        df_error_save[col] = df_error_save[col].fillna('').astype(str)
                    
                    # 저장 시도 - Parquet 실패하면 CSV로 대체
                    try:
                        error_file = os.path.join(checkpoint_dir, f'processed_data_until_error_{error_timestamp}.parquet')
                        df_error_save.to_parquet(error_file)
                    except Exception as e:
                        logger.warning(f"오류 파일 Parquet 저장 실패: {str(e)}. CSV 형식으로 저장합니다.")
                        error_file = os.path.join(checkpoint_dir, f'processed_data_until_error_{error_timestamp}.csv')
                        df_error_save.to_csv(error_file, index=True, encoding='utf-8-sig')
                    
                    logger.info(f"오류 발생 시점까지 처리된 데이터 {len(all_processed_df)}개 행을 {error_file}에 저장했습니다.")
                
                # 오류 발생 시 실패 기록 저장
                if not failed_records.empty:
                    error_failed_file = os.path.join(checkpoint_dir, f'failed_records_until_error_{error_timestamp}.csv')
                    failed_records.to_csv(error_failed_file, index=False, encoding='utf-8-sig')
                    logger.info(f"오류 발생 시점까지 실패한 레코드 {len(failed_records)}개를 {error_failed_file}에 저장했습니다.")
                
                # 예외 재발생
                raise
        
        # 실패한 레코드 저장
        if not failed_records.empty:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            failed_file = os.path.join(checkpoint_dir, f'failed_records_{timestamp}.csv')
            failed_records.to_csv(failed_file, index=False, encoding='utf-8-sig')
            logger.info(f"실패한 레코드 {len(failed_records)}개를 {failed_file}에 저장했습니다.")
        
        # 처리 결과에 대한 통계 로깅
        processed_columns = [col for col in all_processed_df.columns if col.split('_')[0] in classifier.classifiers.keys()]
        
        logger.info("=== 전체 처리 결과 ===")
        logger.info(f"총 레코드 수: {total_rows}")
        logger.info(f"처리된 컬럼: {processed_columns}")
        logger.info(f"실패한 레코드 수: {len(failed_records)}")
        
        # 데이터프레임 사본 생성
        final_df = all_processed_df.copy()
        
        # 타입 충돌 해결을 위한 열 타입 변환
        for col in final_df.columns:
            if col.endswith('_pi_next_schedule'):
                # 모든 _pi_next_schedule 컬럼은 문자열로 통일
                final_df[col] = final_df[col].fillna('').astype(str)
            elif final_df[col].dtype == 'object':
                # 다른 객체 타입의 열도 모두 문자열로 통일
                final_df[col] = final_df[col].fillna('').astype(str)
        
        # 최종 파일 저장 - Parquet 및 CSV 둘 다 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Parquet 저장 시도
        try:
            output_file = f'processed_medical_data_{timestamp}.parquet'
            final_df.to_parquet(output_file)
            logger.info(f"처리된 데이터를 {output_file}에 저장했습니다.")
        except Exception as e:
            logger.error(f"Parquet 저장 실패: {str(e)}")
            
        # 백업용 CSV 저장 (필요 시)
        try:
            csv_output_file = f'processed_medical_data_{timestamp}.csv'
            final_df.to_csv(csv_output_file, index=True, encoding='utf-8-sig')
            logger.info(f"처리된 데이터를 CSV 백업 {csv_output_file}에 저장했습니다.")
        except Exception as e:
            logger.error(f"CSV 저장 실패: {str(e)}")
        
        # 처리가 모두 완료되면 체크포인트 파일 삭제
        if os.path.exists(checkpoint_file):
            os.remove(checkpoint_file)
            logger.info("체크포인트 파일 삭제 완료")
        
        # 임시 파일 삭제 (선택 사항 - 체크포인트 폴더의 이력을 유지하고 싶다면 이 부분은 주석 처리할 수 있습니다)
        for file in os.listdir(checkpoint_dir):
            if file.startswith('temp_processed_data_') and file.endswith('.parquet'):
                try:
                    os.remove(os.path.join(checkpoint_dir, file))
                except Exception as e:
                    logger.warning(f"임시 파일 {file} 삭제 실패: {str(e)}")
        
        logger.info("\n=== 세부 처리 결과 ===")
        for column in all_processed_df.columns:
            if '_' in column:  # 파생 컬럼만 표시
                valid_count = all_processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")
                if all_processed_df[column].dtype in ['object', 'category']:
                    value_counts = all_processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")
                    
    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())  # 상세 오류 정보 출력
        sys.exit(1)
    finally:
        # 전체 실행 시간 측정 종료 및 로깅
        total_end_time = datetime.now()
        total_duration = (total_end_time - total_start_time).total_seconds()
        hours, remainder = divmod(total_duration, 3600)
        minutes, seconds = divmod(remainder, 60)
        
        logger.info(f"전체 실행 시간: {int(hours)}시간 {int(minutes)}분 {seconds:.2f}초")
        if 'total_rows' in locals() and total_duration > 0:
            records_per_second = total_rows / total_duration
            logger.info(f"평균 처리 속도: {records_per_second:.2f}건/초")
        
        logger.info("Program execution completed")
       

if __name__ == "__main__":
    main()

2025-03-03 22:35:06,527 - __main__ - INFO - 원본 데이터 총 28108개를 15개 청크로 분할하여 처리합니다. (청크 크기: 2000)
청크 처리 진행률:   0%|          | 0/15 [00:00<?, ?it/s]2025-03-03 22:35:06,537 - __main__ - INFO - 청크 1/15 처리 시작 (레코드 1-2000)
2025-03-03 22:35:06,547 - __main__ - INFO - Starting medical data processing at 2025-03-03 22:35:06.547169
2025-03-03 22:35:06,547 - __main__ - INFO - Processing column: CC
2025-03-03 22:35:06,580 - __main__ - INFO - Resumed from checkpoint for CC
2025-03-03 22:35:06,580 - __main__ - INFO - Column CC generated 0 derived columns: []
2025-03-03 22:35:06,580 - __main__ - INFO - Processing column: 약
2025-03-03 22:35:06,582 - __main__ - INFO - Processing text at index 1, first 100 chars: 약: 약먹고 나서부터 갈비뼈부터 등까지 근육이 아팠어요, 아직 옆구리 있는 부분이 아파요
2025-03-03 22:35:06,582 - __main__ - INFO - Processing text at index 18, first 100 chars: 약: 세크로 정 저녁만. 페리슨 7일 남았어요
2025-03-03 22:35:06,583 - __main__ - INFO - Processing text at index 19, first 100 chars: 약: 3일먹다가 안아파서 안먹었어요
2025-03-03 22:35:06,5

KeyboardInterrupt: 

2025-03-03 22:49:49,540 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2025-03-03 22:49:49,541 - openai._base_client - INFO - Retrying request to /chat/completions in 1.594487 seconds
2025-03-03 22:49:51,577 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2025-03-03 22:49:51,582 - __main__ - ERROR - API call failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
2025-03-03 22:49:54,136 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2025-03-03 22:49:54,142 - openai._base_client - INFO - Retrying request to /chat/completions in 0.905165 s

In [29]:
df.head(1)

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI
13927,2211-09,2022-12-16,"구강내과#3물리치료 , 증상 ck, 소견서(free)증상: 아픈거 없어요, 의식을 ...",NaN,NaN,"습관: 딱딱하고 질긴거 피하려 했어요, 이랑이 안닿게 했어요",NaN,NaN,12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작...


In [41]:
pd.read_parquet('processed_medical_data_20250303_134711.parquet')

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_clinic_history_desc,CC_factor_habbit,CC_treat_plan,CC_severity,CC_vas,CC_duration,약_medication_type,약_frequency,약_duration,약_compliance,장치_device_type,장치_usage_pattern,장치_duration,장치_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,찜질_status,찜질_frequency,찜질_duration,찜질_method,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method",PI_pi_aggravating_factors,PI_pi_TMJ_PI_desc,PI_pi_TMJ_PI_treatment,PI_pi_drug_treatment,PI_pi_closing_dentalgear_desc,PI_pi_next_schedule,PI_pi_next_ck,PI_pi_PI_diagnosis_jojint,PI_pi_occlusal_treatment,PI_pi_medication_prescription,PI_pi_other_treatment,PI_pi_onset,PI_pi_pattern,PI_pi_status
13927,2211-09,2022-12-16,"구강내과#3물리치료 , 증상 ck, 소견서(free)증상: 아픈거 없어요, 의식을 ...",,,"습관: 딱딱하고 질긴거 피하려 했어요, 이랑이 안닿게 했어요",,,12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작...,왼쪽 턱,딱딱 소리,딱딱 소리가 나는 것,딱딱 소리가 나는 것,보톡스 후 변화 없음,보톡스 맞은지 2주 넘었는데 처음 5일만 힘이 빠지는 것 같더니 지금은 변화가 느껴...,왼쪽에서 딱딱 소리가 나는 것 같아요,하루에 두 번 정도 의식을 해서,"물리치료, 샤워 시 온수찜질 주 2회",NaN,NaN,2주 넘었는데,,,,,,,,,편측성저작,medium,aware,improved,NaN,,NaN,,,,,,,측두하악장애분석검사,"측두하악관절자극요법-단순자극, 측두하악관절자극요법-전기자극, 측두하악관절자극요법-복...",,,21.0,"증상 ck, 근육두께 ck",저작근 장애 (K07.66),,,,3주후,,
20895,2202-242,2022-03-14,구강종합검진이 받고 싶습니다. 이를 많이 악물고입마름 현상이 많이 심해서요. --3...,,,,,,,턱관절,"퇴행성 변화, 통증","턱관절의 통증, 퇴행성 변화","턱관절의 비정상적 움직임, 제한된 개구","이갈이, 입마름, 스트레스로 인한 턱 근육 긴장","구강종합검진을 받고 싶어하며, 과거에 다른 곳에서 검사한 적은 없다고 언급됨.","3년 전부터 입마름 현상이 심해졌고, 턱관절의 퇴행성 변화가 의심되어 파노라마와 C...","이를 많이 악물고, 입마름 현상이 심하며, 잠을 잘 자고 일어나면 입이 가장 많이 ...","측두하악장애분석검사, 파노라마, CT 촬영을 통해 진단받고, 타액검사를 통해 분비율...",NaN,NaN,3년정도,,,,,,,,,,,,,NaN,,NaN,,,,,,,,,,,,,,,,,,,
8259,2309-268,2023-10-04,턱관절10년전에양쪽인데 왼쪽이 심하게 딱딱소리 났어요벌리는게 불편해요강남쪽 구강내과...,,,,,,* daytime/ clenching-> MFP TTH* both ID* Rt DJ...,"턱관절, 양쪽 관자놀이","뻐근감, 통증","입이 잘 안 벌어지고 통증이 있었음, 턱에 힘을 주는 것이 불편함","벌리는 것이 불편함, 턱에 힘을 주는 것이 가장 불편함","스트레스 받을 때 턱에 힘을 주면서 두통 발생, 이 악무는 습관, 잠을 잘 못 잠,...","강남쪽 구강내과에서 물리치료랑 장치 1년정도 착용, 보톡스 맞고 싶어서 내원","턱관절 10년 전에 양쪽인데 왼쪽이 심하게 딱딱 소리, 입이 잘 안 벌어지고 통증,...","이 악무는 습관 O, 딱딱하고 질긴 음식 X, 잠 잘 못자요 5시간 정도 자고 일찍...","보톡스 (100unit) 완료, 초음파 진단, 장치 조정 비용 2회 free, 추후...",NaN,8.0,10년,,,,,,,,,,,,,NaN,,NaN,,,,,,이 악물기,"파노라마, Cone Beam CT, 측두하악장애분석검사, 초음파, 파노라마(특수)","분사신장치료, 물리치료",,보톡스(교합개선 목적),7.0,증상 ck,"퇴행성 관절염 (K07.65), 턱관절 통증 (K07.63), 저작근 장애 (K07...",,,보톡스 시술,,,
2083,2302-65,2024-01-09,"물리치료 , 장치 ck구강내과#10/ 혜련증상: 턱에 힘 주고나면 오른쪽 턱 뻐근해...",,장치: APS 주4회 착용/ 밴드X/ 불편감X,습관: 주 1회 정도 떡볶이 같은 음식 먹었어요/ 치아끼리 안닿게 턱에 힘 풀어요,찜질: 주 3회 온찜질팩 20분 해요,"마사지,스트레칭: 둘다 주3-4회",*Deep bite,"오른쪽 턱, 오른쪽 귀 앞쪽, 왼쪽 턱","뻐근함, 걸리는 느낌, 소리","오른쪽 턱 뻐근함, 오른쪽 귀 앞쪽 걸리는 느낌",입 벌릴 때 왼쪽 턱에서 소리,턱에 힘 주고 나면 뻐근함,"물리치료, 장치 치료","2-3달 전부터 입 벌릴 때 오른쪽 귀 앞쪽이 걸리는 느낌, 왼쪽 턱에서 소리가 나...",턱에 힘을 주면 오른쪽 턱이 뻐근해짐,"물리치료, 장치 치료로 증상 강도와 빈도 감소 (VAS 5->1)",1.0,1.0,2-3달 전,,,,,APS,partial,,good,편측성저작,medium,aware,improved,1.0,medium,20.0,hot,both,medium,,,,,,,,,,,,,,,,
25638,2206-260,2022-11-25,"내과 #6물리치료 , 장치 ck경북대 안감증상: 요즘 딱히 불편감 없ㅇ ㅣ지내고 있...",약: 처방된건 다 복용,,,찜질: 매일 5~10분 / 턱관절찜질팩,,* 교합변화 가능성 고지,오른쪽 턱,소리,가끔 오른쪽에서 소리가 남,음식 씹을 때 소리가 남,스트레스로 인한 증상 없음,"물리치료, 장치 치료","경북대에서 치료받음, 현재 증상 없음","음식 씹을 때 오른쪽에서 소리, 질기고 딱딱한 음식 피함, 이 닿지 않도록 신경 씀","장치 매일 착용, 장치 불편 없음, 습관 조절",NaN,NaN,,,regular,,good,,,,,,,,,1.0,high,10.0,hot,,,,,,,,,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13965,2211-109,2023-11-28,"#26,27 in prep 가능성 (일반 처음), 일반: 1달전쯤에 집앞 치과에 검...",,"장치: aps 주6일 착용, 불편감 x, 밴드 x.","습관:질긴음식 x, 치아끼리 안닿게o",찜질: 매일 15-20분,"마사지,스트레칭: 매일샤워시",[완료] #27 : Vericom inlay(caries (M)-endo가능성)[취...,턱,"딱소리, 모래갈리는 소리, 막힌 느낌","턱 내밀었다가 벌리면 왼쪽 딱소리 작게, 오른쪽 모래갈리는 소리",그냥 입벌리면 왼쪽 딱소리 크게나고 오른쪽은 막힌 느낌,통증이나 뻐근함은 없었음,"1달 전 집 앞 치과에서 왼쪽 위 치아 두 개가 썩어서 치료 필요, 한 개는 충치가...","턱 내밀었다가 벌리면 왼쪽 딱소리 작게, 오른쪽 모래갈리는 소리, 그냥 입 벌리면 ...",특정 습관이나 자